# Chapter 2: Common Functional Groups in Drugs

## 1. Introduction

Functional groups are the reactive centers of organic molecules that determine their chemical properties, biological activity, and pharmacological behavior. In drug design, understanding which functional groups are beneficial and which should be avoided is fundamental to developing safe and effective therapeutics.

Approximately 85% of FDA-approved small molecule drugs contain nitrogen-containing functional groups, while specific reactive functionalities are systematically excluded due to toxicity or promiscuity.This module will equip you with the knowledge to identify, analyze, and evaluate functional groups in drug candidates using computational tools. You'll learn both the "privileged" functional groups commonly found in drugs and the "liability" groups that flag potential safety concerns.

---

## 2. Key Concepts and Definitions

* **Functional Group:** A specific group of atoms within a molecule responsible for characteristic chemical reactions and properties.
* **Pharmacophore:** The ensemble of functional groups and spatial features in a molecule responsible for its biological activity.
* **Toxicophore:** Structural features associated with toxicity, including metabolic activation to reactive intermediates or direct cellular damage.
* **PAINS (Pan-Assay Interference Compounds):** Substructures that show frequent false positives in biological assays through non-specific reactivity.
* **Lipinski's Rule of Five:** Guidelines suggesting drug-like molecules should have ≤5 H-bond donors, ≤10 H-bond acceptors, molecular weight ≤500 Da, and LogP ≤5. Functional groups directly impact these parameters.
* **ADMET:** Absorption, Distribution, Metabolism, Excretion, and Toxicity—all influenced by the functional groups present in a molecule.

---

## 3. Main Content

This section provides a detailed breakdown of key functional groups, their properties, and their significance in drug molecules.

### 3.1 Oxygen-Containing Groups

In [1]:
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display, HTML
import json
import uuid

# --- 1. Define Data: Functional Group Info + Drug Info ---

fg_data = {
    'Hydroxyl (-OH)': {
        'smarts': '[OX2H]', 
        'drug_name': 'Paracetamol (Acetaminophen)',
        'drug_smiles': 'CC(=O)Nc1ccc(O)cc1',
        'fg_desc': """
            <b>Functional Group Properties:</b><br>
            Hydroxyl groups improve water solubility (polar) and act as both hydrogen bond donors and acceptors. 
            In metabolism, they often serve as a "handle" for conjugation (Phase II metabolism).
        """,
        'drug_desc': """
            <b>About the Drug:</b><br>
            Paracetamol is a widely used analgesic (pain reliever) and antipyretic (fever reducer). 
            Unlike NSAIDs, it has little anti-inflammatory activity.
        """
    },
    'Carboxylic Acid (-COOH)': {
        'smarts': '[CX3](=O)[OX2H1]', 
        'drug_name': 'Ibuprofen',
        'drug_smiles': 'CC(C)Cc1ccc(C(C)C(=O)O)cc1',
        'fg_desc': """
            <b>Functional Group Properties:</b><br>
            This acidic group (pKa ≈ 4-5) is usually ionized (negatively charged) at physiological pH. 
            This charge allows for strong electrostatic interactions (salt bridges) with protein targets.
        """,
        'drug_desc': """
            <b>About the Drug:</b><br>
            Ibuprofen is a non-steroidal anti-inflammatory drug (NSAID). It works by inhibiting the 
            COX enzymes, which convert arachidonic acid into prostaglandins (mediators of pain/inflammation).
        """
    },
    'Ester (-COOR)': {
        'smarts': '[#6][CX3](=O)[OX2H0][#6]', 
        'drug_name': 'Aspirin (Acetylsalicylic acid)',
        'drug_smiles': 'CC(=O)Oc1ccccc1C(=O)O',
        'fg_desc': """
            <b>Functional Group Properties:</b><br>
            Esters are often used in prodrug design. They are lipophilic (crossing membranes easily) 
            but are liable to hydrolysis by esterases in the blood/liver to release the active acid.
        """,
        'drug_desc': """
            <b>About the Drug:</b><br>
            Aspirin is used to reduce pain, fever, and inflammation. Uniquely, it also acts as an 
            antiplatelet agent by <i>irreversibly</i> acetylating the COX enzyme (due to this specific ester group).
        """
    },
    'Ketone (-C=O)': {
        'smarts': '[#6][CX3](=O)[#6]', 
        'drug_name': 'Progesterone',
        'drug_smiles': 'CC(=O)C1CCC2C1(CCC3C2CCC4=CC(=O)CCC34C)C',
        'fg_desc': """
            <b>Functional Group Properties:</b><br>
            Ketones are polar and act as hydrogen bond acceptors (but not donors). They are rigid planar 
            groups often found in steroid scaffolds.
        """,
        'drug_desc': """
            <b>About the Drug:</b><br>
            Progesterone is an endogenous steroid hormone involved in the menstrual cycle, pregnancy, 
            and embryogenesis. It binds to nuclear progesterone receptors to regulate gene expression.
        """
    },
    'Ether (-O-)': {
        'smarts': '[OD2]([#6])[#6]', 
        'drug_name': 'Fluoxetine (Prozac)',
        'drug_smiles': 'CNCCC(C1=CC=CC=C1)OC2=CC=C(C=C2)C(F)(F)F',
        'fg_desc': """
            <b>Functional Group Properties:</b><br>
            Ethers are chemically stable linkers. They provide flexibility to the molecule and can act 
            as weak hydrogen bond acceptors. They are generally resistant to metabolic hydrolysis.
        """,
        'drug_desc': """
            <b>About the Drug:</b><br>
            Fluoxetine is a Selective Serotonin Reuptake Inhibitor (SSRI) antidepressant. 
            It treats major depressive disorder, obsessive-compulsive disorder (OCD), and panic disorder.
        """
    }
}

# --- 2. Generate Images and Organize Data ---
svg_library = {}
fg_text_library = {}
drug_text_library = {}

for name, data in fg_data.items():
    mol = Chem.MolFromSmiles(data['drug_smiles'])
    pattern = Chem.MolFromSmarts(data['smarts'])
    
    # Find atoms to highlight
    matches = mol.GetSubstructMatches(pattern)
    atoms_to_highlight = [atom_idx for match in matches for atom_idx in match]
    
    # Draw
    drawer = rdMolDraw2D.MolDraw2DSVG(500, 300)
    opts = drawer.drawOptions()
    opts.clearBackground = False
    opts.padding = 0.1
    
    # Highlight color (Cyan)
    opts.setHighlightColour((0.2, 1.0, 1.0)) 
    
    drawer.DrawMolecule(mol, highlightAtoms=atoms_to_highlight, highlightBonds=[])
    drawer.FinishDrawing()
    
    svg_library[name] = drawer.GetDrawingText()
    fg_text_library[name] = data['fg_desc']
    drug_text_library[name] = f"<strong>{data['drug_name']}</strong><br>{data['drug_desc']}"

# --- 3. HTML Interface ---

unique_id = str(uuid.uuid4())[:8]
viewer_id = f"viewer_{unique_id}"
fg_desc_id = f"fg_desc_{unique_id}"
drug_desc_id = f"drug_desc_{unique_id}"

# Serialize for JS
svg_json = json.dumps(svg_library)
fg_json = json.dumps(fg_text_library)
drug_json = json.dumps(drug_text_library)

buttons_html = ""
for name in fg_data.keys():
    buttons_html += (
        f'<button onclick="window.showLesson_{unique_id}(\'{name}\')" '
        f'style="display:block; width:100%; margin: 6px 0; padding: 12px; '
        f'text-align:left; background: #ffffff; border: 1px solid #ddd; '
        f'border-radius: 6px; cursor: pointer; font-weight:600; color: #333; transition: 0.2s;">'
        f'{name}</button>'
    )

html_code = f"""
<div style="display: flex; font-family: 'Segoe UI', sans-serif; border: 1px solid #ccc; border-radius: 8px; overflow: hidden; max-width: 900px; background-color: #f9f9f9;">
    
    <div style="width: 25%; background-color: #f0f2f5; padding: 15px; border-right: 1px solid #ddd;">
        <h4 style="margin-top:0; color: #444;">Select Functional Group</h4>
        {buttons_html}
    </div>

    <div style="width: 75%; padding: 20px; display: flex; flex-direction: column;">
        
        <div id="{fg_desc_id}" style="background-color: #e8f4fd; border-left: 4px solid #2196F3; padding: 12px; border-radius: 4px; color: #0d47a1; margin-bottom: 15px; font-size: 0.95em;">
            Select a group to begin.
        </div>
        
        <div id="{viewer_id}" style="flex-grow: 1; background: white; border: 1px solid #eee; border-radius: 8px; display: flex; align-items: center; justify-content: center; min-height: 300px; margin-bottom: 15px;">
            </div>
        
        <div id="{drug_desc_id}" style="background-color: #e8f5e9; border-left: 4px solid #4caf50; padding: 12px; border-radius: 4px; color: #1b5e20; font-size: 0.95em;">
            </div>
        
    </div>
</div>

<script>
    (function() {{
        const svgs = {svg_json};
        const fg_descs = {fg_json};
        const drug_descs = {drug_json};
        
        const viewer = document.getElementById('{viewer_id}');
        const fgBox = document.getElementById('{fg_desc_id}');
        const drugBox = document.getElementById('{drug_desc_id}');

        window.showLesson_{unique_id} = function(name) {{
            // Update Content
            fgBox.innerHTML = fg_descs[name];
            viewer.innerHTML = svgs[name];
            drugBox.innerHTML = drug_descs[name];
        }}
        
        // Initialize with first entry
        const firstKey = Object.keys(svgs)[0];
        if(firstKey) window.showLesson_{unique_id}(firstKey);
    }})();
</script>
"""

display(HTML(html_code))

### 3.2 Nitrogen-Containing Groups

In [2]:
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display, HTML
import json
import uuid

# --- 1. Define Data: Functional Group Info + Drug Info ---

fg_data = {
    'Amine (-NH₂, -NHR, -NR₂)': {
        # SMARTS: Matches Nitrogen not attached to a Carbonyl (excludes amides)
        'smarts': '[NX3;!$(NC=O);!$(NC=S)]', 
        'drug_name': 'Diphenhydramine (Benadryl)',
        'drug_smiles': 'CN(C)CCOC(C1=CC=CC=C1)C2=CC=CC=C2',
        'fg_desc': """
            <b>Functional Group Properties:</b><br>
            Amines are the most prevalent functional group in FDA-approved drugs (~75% contain at least one nitrogen). 
            They provide basicity for salt formation (improving solubility), hydrogen bonding for target interactions, 
            and metabolic stability.
        """,
        'drug_desc': """
            <b>About the Drug:</b><br>
            Diphenhydramine is a first-generation antihistamine. The tertiary amine group is crucial for its interaction 
            with histamine H1 receptors. It is widely used to treat allergies, insomnia, and motion sickness.
        """
    },
    'Amide (-C(=O)NH-)': {
        # SMARTS: Carbonyl Carbon attached to Nitrogen
        'smarts': '[CX3](=O)[NX3]', 
        'drug_name': 'Lidocaine',
        'drug_smiles': 'CCN(CC)CC(=O)NC1=C(C)C=CC=C1C',
        'fg_desc': """
            <b>Functional Group Properties:</b><br>
            Amides combine the hydrogen bonding capacity of both carbonyl and NH groups while providing metabolic 
            stability (resistant to hydrolysis compared to esters). The amide bond's planarity restricts conformational 
            flexibility, which can be advantageous for target selectivity.
        """,
        'drug_desc': """
            <b>About the Drug:</b><br>
            Lidocaine is a local anesthetic and antiarrhythmic drug. The amide linkage in the center of the molecule 
            is chemically stable, preventing it from breaking down too quickly in the body, unlike earlier ester-based anesthetics (like procaine).
        """
    },
    'Heterocyclic Nitrogen': {
        # SMARTS: Nitrogen inside a ring structure
        'smarts': '[#7;R]', 
        'drug_name': 'Nicotine',
        'drug_smiles': 'CN1CCCC1C2=CN=CC=C2',
        'fg_desc': """
            <b>Functional Group Properties:</b><br>
            Nitrogen atoms embedded in rings (heterocycles) are extremely common drug scaffolds. 
            Examples of such rings are pyridine, pyrrole and indole. They often dictate the overall shape 
            and electronic properties of the drug.
        """,
        'drug_desc': """
            <b>About the Drug:</b><br>
            Nicotine contains two heterocyclic rings: a pyridine (aromatic, right) and a pyrrolidine (aliphatic, left). 
            It acts as an agonist at nicotinic acetylcholine receptors (nAChRs) in the central nervous system.
        """
    },
    'Sulfonamide (-SO₂NR₂)': {
        # SMARTS: Sulfur double bonded to 2 Oxygens and single bonded to Nitrogen
        'smarts': '[#16X4](=[OX1])(=[OX1])[#7X3]', 
        'drug_name': 'Sulfamethoxazole',
        'drug_smiles': 'Cc1cc(NS(=O)(=O)c2ccc(N)cc2)no1',
        'fg_desc': """
            <b>Functional Group Properties:</b><br>
            This group is a weak acid (pKa ≈ 9-10) and a strong hydrogen bond acceptor. 
            It is often used as a bioisostere for carboxylic acids and provides unique geometry for binding pockets.
        """,
        'drug_desc': """
            <b>About the Drug:</b><br>
            Sulfamethoxazole is a bacteriostatic antibiotic. It works by mimicking PABA (para-aminobenzoic acid) 
            and inhibiting the synthesis of dihydrofolic acid in bacteria. The sulfonamide group is the key structural mimic.
        """
    }
}

# --- 2. Generate Images and Organize Data ---
svg_library = {}
fg_text_library = {}
drug_text_library = {}

for name, data in fg_data.items():
    mol = Chem.MolFromSmiles(data['drug_smiles'])
    pattern = Chem.MolFromSmarts(data['smarts'])
    
    # Find atoms to highlight
    matches = mol.GetSubstructMatches(pattern)
    atoms_to_highlight = [atom_idx for match in matches for atom_idx in match]
    
    # Draw
    drawer = rdMolDraw2D.MolDraw2DSVG(500, 300)
    opts = drawer.drawOptions()
    opts.clearBackground = False
    opts.padding = 0.1
    
    # Highlight color (Cyan)
    opts.setHighlightColour((0.2, 1.0, 1.0)) 
    
    drawer.DrawMolecule(mol, highlightAtoms=atoms_to_highlight, highlightBonds=[])
    drawer.FinishDrawing()
    
    svg_library[name] = drawer.GetDrawingText()
    fg_text_library[name] = data['fg_desc']
    drug_text_library[name] = f"<strong>{data['drug_name']}</strong><br>{data['drug_desc']}"

# --- 3. HTML Interface ---

unique_id = str(uuid.uuid4())[:8]
viewer_id = f"viewer_{unique_id}"
fg_desc_id = f"fg_desc_{unique_id}"
drug_desc_id = f"drug_desc_{unique_id}"

# Serialize for JS
svg_json = json.dumps(svg_library)
fg_json = json.dumps(fg_text_library)
drug_json = json.dumps(drug_text_library)

buttons_html = ""
for name in fg_data.keys():
    buttons_html += (
        f'<button onclick="window.showLesson_{unique_id}(\'{name}\')" '
        f'style="display:block; width:100%; margin: 6px 0; padding: 12px; '
        f'text-align:left; background: #ffffff; border: 1px solid #ddd; '
        f'border-radius: 6px; cursor: pointer; font-weight:600; color: #333; transition: 0.2s;">'
        f'{name}</button>'
    )

html_code = f"""
<div style="display: flex; font-family: 'Segoe UI', sans-serif; border: 1px solid #ccc; border-radius: 8px; overflow: hidden; max-width: 900px; background-color: #f9f9f9;">
    
    <div style="width: 25%; background-color: #f0f2f5; padding: 15px; border-right: 1px solid #ddd;">
        <h4 style="margin-top:0; color: #444;">Select Functional Group</h4>
        {buttons_html}
    </div>

    <div style="width: 75%; padding: 20px; display: flex; flex-direction: column;">
        
        <div id="{fg_desc_id}" style="background-color: #e8f4fd; border-left: 4px solid #2196F3; padding: 12px; border-radius: 4px; color: #0d47a1; margin-bottom: 15px; font-size: 0.95em;">
            Select a group to begin.
        </div>
        
        <div id="{viewer_id}" style="flex-grow: 1; background: white; border: 1px solid #eee; border-radius: 8px; display: flex; align-items: center; justify-content: center; min-height: 300px; margin-bottom: 15px;">
            </div>
        
        <div id="{drug_desc_id}" style="background-color: #e8f5e9; border-left: 4px solid #4caf50; padding: 12px; border-radius: 4px; color: #1b5e20; font-size: 0.95em;">
            </div>
        
    </div>
</div>

<script>
    (function() {{
        const svgs = {svg_json};
        const fg_descs = {fg_json};
        const drug_descs = {drug_json};
        
        const viewer = document.getElementById('{viewer_id}');
        const fgBox = document.getElementById('{fg_desc_id}');
        const drugBox = document.getElementById('{drug_desc_id}');

        window.showLesson_{unique_id} = function(name) {{
            // Update Content
            fgBox.innerHTML = fg_descs[name];
            viewer.innerHTML = svgs[name];
            drugBox.innerHTML = drug_descs[name];
        }}
        
        // Initialize with first entry
        const firstKey = Object.keys(svgs)[0];
        if(firstKey) window.showLesson_{unique_id}(firstKey);
    }})();
</script>
"""

display(HTML(html_code))

### 3.3 Functional Groups to Avoid: Structural Alerts and Toxicophores

In [3]:
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display, HTML
import json
import uuid

# --- 1. Define Data: Functional Group Info + Drug Info ---

fg_data = {
    'Halogens (F, Cl, Br, I)': {
        # SMARTS: Matches Fluorine, Chlorine, Bromine, or Iodine
        'smarts': '[F,Cl,Br,I]', 
        'drug_name': 'Haloperidol',
        'drug_smiles': 'O=C(CCCCN1CCC(O)(c2ccc(Cl)cc2)CC1)c3ccc(F)cc3',
        'fg_desc': """
            <b>Functional Group Properties:</b><br>
            Halogens are used to modify a drug's lipophilicity (fat-solubility) to help cross membranes (like the blood-brain barrier). 
            They are also used to block sites of metabolic breakdown (e.g., adding Fluorine to prevent oxidation) and can form halogen bonds.
        """,
        'drug_desc': """
            <b>About the Drug:</b><br>
            Haloperidol is a typical antipsychotic used to treat schizophrenia. It contains both a <b>Fluorine</b> and a <b>Chlorine</b> atom. 
            The Fluorine aids in potency and metabolic stability, while the lipophilic nature of the molecule allows it to enter the CNS.
        """
    },
    'Thiol (-SH)': {
        # SMARTS: Sulfur with exactly one Hydrogen attached
        'smarts': '[#16;H1]', 
        'drug_name': 'Captopril',
        'drug_smiles': 'C[C@H](CS)C(=O)N1CCC[C@H]1C(=O)O',
        'fg_desc': """
            <b>Functional Group Properties:</b><br>
            The thiol group is weakly acidic (pKa ≈ 9-10) and a potent nucleophile. Crucially in medicinal chemistry, 
            it is an excellent <b>chelator of metal ions</b> (like Zinc or Copper) within enzyme active sites.
        """,
        'drug_desc': """
            <b>About the Drug:</b><br>
            Captopril is an ACE inhibitor for hypertension. It was rationally designed so that its Thiol (-SH) group 
            would form a direct coordinate bond with the <b>Zinc ion</b> present in the Angiotensin-Converting Enzyme, thereby disabling the enzyme.
        """
    }
}

# --- 2. Generate Images and Organize Data ---
svg_library = {}
fg_text_library = {}
drug_text_library = {}

for name, data in fg_data.items():
    mol = Chem.MolFromSmiles(data['drug_smiles'])
    pattern = Chem.MolFromSmarts(data['smarts'])
    
    # Find atoms to highlight
    matches = mol.GetSubstructMatches(pattern)
    atoms_to_highlight = [atom_idx for match in matches for atom_idx in match]
    
    # Draw
    drawer = rdMolDraw2D.MolDraw2DSVG(500, 300)
    opts = drawer.drawOptions()
    opts.clearBackground = False
    opts.padding = 0.1
    
    # Highlight color (Cyan)
    opts.setHighlightColour((0.2, 1.0, 1.0)) 
    
    drawer.DrawMolecule(mol, highlightAtoms=atoms_to_highlight, highlightBonds=[])
    drawer.FinishDrawing()
    
    svg_library[name] = drawer.GetDrawingText()
    fg_text_library[name] = data['fg_desc']
    drug_text_library[name] = f"<strong>{data['drug_name']}</strong><br>{data['drug_desc']}"

# --- 3. HTML Interface ---

unique_id = str(uuid.uuid4())[:8]
viewer_id = f"viewer_{unique_id}"
fg_desc_id = f"fg_desc_{unique_id}"
drug_desc_id = f"drug_desc_{unique_id}"

# Serialize for JS
svg_json = json.dumps(svg_library)
fg_json = json.dumps(fg_text_library)
drug_json = json.dumps(drug_text_library)

buttons_html = ""
for name in fg_data.keys():
    buttons_html += (
        f'<button onclick="window.showLesson_{unique_id}(\'{name}\')" '
        f'style="display:block; width:100%; margin: 6px 0; padding: 12px; '
        f'text-align:left; background: #ffffff; border: 1px solid #ddd; '
        f'border-radius: 6px; cursor: pointer; font-weight:600; color: #333; transition: 0.2s;">'
        f'{name}</button>'
    )

html_code = f"""
<div style="display: flex; font-family: 'Segoe UI', sans-serif; border: 1px solid #ccc; border-radius: 8px; overflow: hidden; max-width: 900px; background-color: #f9f9f9;">
    
    <div style="width: 25%; background-color: #f0f2f5; padding: 15px; border-right: 1px solid #ddd;">
        <h4 style="margin-top:0; color: #444;">Select Group</h4>
        {buttons_html}
    </div>

    <div style="width: 75%; padding: 20px; display: flex; flex-direction: column;">
        
        <div id="{fg_desc_id}" style="background-color: #e8f4fd; border-left: 4px solid #2196F3; padding: 12px; border-radius: 4px; color: #0d47a1; margin-bottom: 15px; font-size: 0.95em;">
            Select a functional group to begin.
        </div>
        
        <div id="{viewer_id}" style="flex-grow: 1; background: white; border: 1px solid #eee; border-radius: 8px; display: flex; align-items: center; justify-content: center; min-height: 300px; margin-bottom: 15px;">
            </div>
        
        <div id="{drug_desc_id}" style="background-color: #e8f5e9; border-left: 4px solid #4caf50; padding: 12px; border-radius: 4px; color: #1b5e20; font-size: 0.95em;">
            </div>
        
    </div>
</div>

<script>
    (function() {{
        const svgs = {svg_json};
        const fg_descs = {fg_json};
        const drug_descs = {drug_json};
        
        const viewer = document.getElementById('{viewer_id}');
        const fgBox = document.getElementById('{fg_desc_id}');
        const drugBox = document.getElementById('{drug_desc_id}');

        window.showLesson_{unique_id} = function(name) {{
            // Update Content
            fgBox.innerHTML = fg_descs[name];
            viewer.innerHTML = svgs[name];
            drugBox.innerHTML = drug_descs[name];
        }}
        
        // Initialize with first entry
        const firstKey = Object.keys(svgs)[0];
        if(firstKey) window.showLesson_{unique_id}(firstKey);
    }})();
</script>
"""

display(HTML(html_code))

> **Note:** There are certain functional groups that a medicial chemist would avoid when designing drugs. For example,  functional groups like aldehydes can react with lysine residues in proteins, leading to protein modification and potential toxicity. Aldehydes are highly reactive and metabolically unstable.

---

## 4. Summary and Key Takeaways

In this section, we've explored the most common functional groups that determine the properties of drug molecules. 

*   **Functional groups define a drug's identity.** They dictate properties like acidity/basicity, solubility, and the ability to form hydrogen bonds or salt bridges.
*   **Oxygen-containing groups** like hydroxyls, carboxylic acids, and esters are key modulators of polarity and are often involved in hydrogen bonding.
*   **Real-world drug design** strategically employs functional groups to solve problems like poor bioavailability (prodrugs), metabolic instability (metabolic blockers), and weak target binding (hydrogen bond donors/acceptors).

Understanding these fundamental concepts is essential as we move forward to explore more complex aspects of drug structure, such as stereochemistry, which adds a three-dimensional layer to molecular interactions.